# Readability analysis performance scripts:
This notebook consists of two scripts: 
- "Readability analysis performance measuring" for measuring the performance of LLM classification against the ground truth using metrics: MAEm and Krippendorff's alpha for calculating the differences in scoring, and Average Recall to see whether or not the LLM has chosen "N/A" correctly.
- "Readability analysis model comparison" for creating a histogram comparing performance of all used model configurations

**Readability analysis performance measuring script:**

In [1]:
import pandas as pd
import sys
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, mean_absolute_error
import krippendorff
import warnings

# Set font and do not output graphs in Jupyter notebook
matplotlib.use('Agg')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.labelsize'] = 12

Configuration:

In [ ]:
# File names and directories
ROOT = "C:--FILL IN YOUR ROOT FOLDER--"  # root project folder
RDBL_DIR = f"{ROOT}/LLM Analyses/Readability Analysis/Ground Truth sample - rdbl/Readability analyses"
GROUND_TRUTH_DIR = f"{ROOT}/Ground Truth/Annotations"
OUTPUT_DIR = f"{ROOT}/LLM Analyses/Readability Analysis/Ground Truth sample - rdbl/Performance measuring"

GROUND_TRUTH_FILE = "Ground truth - rdbl.csv"  # Manually classified csv
OUTPUT_METRICS = "Rdbl performance"
OUTPUT_SCORE_MATRICES = "Score matrices/Rdbl score matrices"

# edited Shimamura et al. (2025) categories
CRITERIA = ["Crit_A", "Crit_B", "Crit_C", "Crit_D", "Crit_L"]
SUBCRITERIA = [
    "A-01", "A-02", 
    "B-01", "B-02", "B-03", 
    "C-01", "C-02", "C-04", "C-05", "C-06", 
    "D-01",
    "L-01"
]

Main algorithm:

In [ ]:
def load_and_prepare_data(rdbl_file_name):
    """Loads the files and aligns them using the 'Link' column."""
    rdbl_file = os.path.join(RDBL_DIR, rdbl_file_name)
    gt_file = os.path.join(GROUND_TRUTH_DIR, GROUND_TRUTH_FILE)

    if not os.path.exists(rdbl_file): sys.exit(f"LLM input file is missing: {rdbl_file_name}")        
    if not os.path.exists(gt_file): sys.exit(f"Ground Truth input file is missing: {GROUND_TRUTH_FILE}")  

    df_pred = pd.read_csv(rdbl_file)
    df_true = pd.read_csv(gt_file)

    # Clean column names to avoid whitespace issues
    df_pred.columns = df_pred.columns.str.strip()
    df_true.columns = df_true.columns.str.strip()

    # Align data by sorting by Link
    df_pred = df_pred.sort_values(by="URL").reset_index(drop=True)
    df_true = df_true.sort_values(by="URL").reset_index(drop=True)

    if len(df_pred) != len(df_true):
        print(f"WARNING: Row counts do not match ({len(df_pred)} vs {len(df_true)}). Performing inner join...")
        combined = pd.merge(df_true, df_pred, on="URL", suffixes=('_true', '_pred'))
        return combined
    
    return pd.concat([df_true.add_suffix('_true'), df_pred.add_suffix('_pred')], axis=1)

def df_check(rdbl_file_name):
    """Checks if the LLM dataframe contains any error messages."""
    df = pd.read_csv(os.path.join(RDBL_DIR, rdbl_file_name))
    for idx, value in enumerate(df["Gunning_Fog"]):
        val_clean = str(value).strip().upper()
        if "ERROR" in val_clean:
            print(f"{rdbl_file_name} [Row {idx}]: contains ERROR message")

def get_numeric_values(series):
    """Transforms scores 1-5 to floats. N/A becomes NaN."""
    return pd.to_numeric(series.astype(str).str.strip(), errors='coerce')

def get_na_binary(series):
    """Transforms to 1 if value is 'N/A', 0 when it's a valid score (1-5)"""
    is_empty = pd.isna(series)  # nan
    is_na_string = series.astype(str).str.strip().str.upper() == 'N/A'  # 'N/A'
    return (is_empty | is_na_string).astype(int)

def find_column(df_cols, target_base, suffix):
    """Finds a column name in a list regardless of case sensitivity."""
    normalized_target = f"{target_base}_{suffix}".lower().replace(" ", "_")
    for col in df_cols:
        if col.lower().replace(" ", "_") == normalized_target:
            return col
    return None

def measure_performance(rdbl_file, output_metrics, output_score_matrices):

    # Ignore specific RuntimeWarnings in single-class cases
    warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value encountered in scalar divide")
    
    data = load_and_prepare_data(rdbl_file)
    
    results = []    

    # Lists for global calculations over subcriteria
    global_true_scores = []
    global_pred_scores = []
    global_true_na_status = []
    global_pred_na_status = []

    # All targets for individual reporting
    target_list = [(crit, crit, "Main") for crit in CRITERIA] + [(sub, f"{sub.replace("-","_")}_score", "Sub") for sub in SUBCRITERIA]

    for display_name, base_name, crit_type in target_list:
        # Look for the true and predicted versions in the combined dataframe
        true_col = find_column(data.columns, base_name, "true")
        pred_col = find_column(data.columns, base_name, "pred")

        if not true_col or not pred_col:
            print(f"Skipping {display_name}: Not found.")
            continue
        
        # Metric 1: (non-)"N/A" scoring metrics: confusion matrix and average recall
        y_true_na = get_na_binary(data[true_col])   # Convert to binary: N/A = 1 , non-N/A = 0
        y_pred_na = get_na_binary(data[pred_col])

        na_tn, na_fp, na_fn, na_tp = confusion_matrix(y_true_na, y_pred_na, labels=[0, 1]).ravel()

        # Recall on individual level
        recall_na = na_tp / (na_tp + na_fn) if (na_tp + na_fn) > 0 else 0.0
        recall_score = na_tn / (na_tn + na_fp) if (na_tn + na_fp) > 0 else 0.0
        avg_recall = 0.5 * (recall_na + recall_score)

        # Metric 2: MAE & Krippendorff's Alpha for numeric scores (without "N/A" values)
        # Transform scores to numeric values
        y_true_num = get_numeric_values(data[true_col])
        y_pred_num = get_numeric_values(data[pred_col])

        # Filter out N/A values for MAE (for both ground truth and LLM analysis)
        valid_mask = y_true_num.notna() & y_pred_num.notna()
        y_true_clean = y_true_num[valid_mask]
        y_pred_clean = y_pred_num[valid_mask]

        # Base values of nan in case of lack of data
        mae = mean_absolute_error(y_true_clean, y_pred_clean) if len(y_true_clean) > 0 else np.nan
        mean_true = y_true_clean.mean() if len(y_true_clean) > 0 else np.nan
        mean_pred = y_pred_clean.mean() if len(y_true_clean) > 0 else np.nan

        # Krippendorff's Alpha requires a 2D matrix with reliability data per 'annotator'
        # Uses raw numeric data (including nan for "N/A")
        k_alpha = np.nan
        try:
            # Using ordinal scales
            reliability_matrix = np.vstack([y_true_num.values, y_pred_num.values])
            k_alpha = krippendorff.alpha(reliability_data=reliability_matrix, level_of_measurement='ordinal')
        except Exception as e:
            pass # In case of lack of variance to calculate Alpha

        # In case of empty segments
        if len(y_true_clean) == 0:
            mae, k_alpha, mean_true, mean_pred = np.nan, np.nan, np.nan, np.nan
        
        # Save results per (sub)criteria
        results.append({
            "Type": crit_type,
            "Criteria": display_name,
            "Average_Recall_NA": avg_recall,
            "NA_TP": int(na_tp),
            "NA_FP": int(na_fp),
            "NA_TN": int(na_tn),
            "NA_FN": int(na_fn),
            "MAE": mae,
            "Krippendorff_Alpha": k_alpha,
            "Mean_GT": mean_true,
            "Mean_LLM": mean_pred,
            "Valid_Samples": int(len(y_true_clean))
        })

        # Collect data for global subcriteria statistics
        if crit_type == "Sub":
            global_true_scores.extend(y_true_clean.values)
            global_pred_scores.extend(y_pred_clean.values)
            global_true_na_status.extend(y_true_na.values)
            global_pred_na_status.extend(y_pred_na.values)

    # Global Krippendorff's Alpha and macro MAE are based ONLY on subcriteria
    if global_true_scores:
        df_results = pd.DataFrame(results)
        maem = df_results[df_results['Type'] == 'Sub']['MAE'].mean()
        
        # Global average recall
        g_tn, g_fp, g_fn, g_tp = confusion_matrix(global_true_na_status, global_pred_na_status, labels=[0, 1]).ravel()
        g_recall_na = g_tp / (g_tp + g_fn) if (g_tp + g_fn) > 0 else 0.0
        g_recall_score = g_tn / (g_tn + g_fp) if (g_tn + g_fp) > 0 else 0.0
        global_avg_recall = 0.5 * (g_recall_na + g_recall_score)

        # Global MAE without "N/A"
        clean_gt = np.array(global_true_scores)[~np.isnan(global_true_scores) & ~np.isnan(global_pred_scores)]
        clean_pred = np.array(global_pred_scores)[~np.isnan(global_true_scores) & ~np.isnan(global_pred_scores)]
        global_mae = mean_absolute_error(clean_gt, clean_pred)

        # Global Krippendorff's Alpha
        global_matrix = np.vstack([global_true_scores, global_pred_scores])
        global_k_alpha = np.nan
        try:
            global_k_alpha = krippendorff.alpha(reliability_data=global_matrix, level_of_measurement='ordinal')
        except:
            pass

        # Add summary row to .csv
        summary_row = pd.DataFrame([{
            "Criteria": "SUBCRITERIA_GLOBAL_METRICS",
            "Mean_GT": np.mean(global_true_scores),
            "Mean_LLM": np.mean(global_pred_scores),
            "NA_TP": int(g_tp),
            "NA_FP": int(g_fp),
            "NA_TN": int(g_tn),
            "NA_FN": int(g_fn),
            "MAE": global_mae,
            "Krippendorff_Alpha": global_k_alpha,
            "Note": f"MAEm: {maem:.4f} | Global Avg Recall N/A: {global_avg_recall:.4f} | Global Alpha: {global_k_alpha:.4f}"
        }])
        
        pd.concat([df_results, summary_row], ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, output_metrics), index=False, encoding='utf-8-sig')

        # --- VISUALIZATION ---
        # Dutch so graph can be used in thesis scripture. Comparing the averages of subcriteria between ground truth and LLM
        df_sub = df_results[df_results["Type"] == "Sub"].rename(columns={'Mean_GT': 'Ground Truth', 'Mean_LLM': 'LLM'})

        plt.figure(figsize=(12, 6))
        sns.set_style("whitegrid")
        
        # Transform to 'long format'
        df_means = df_sub.melt(id_vars=["Criteria"], value_vars=["Ground Truth", "LLM"], 
                                   var_name="", value_name="Average_score")
        
        # Create barplot
        ax = sns.barplot(data=df_means, x="Criteria", y="Average_score", hue="", palette=["#1E64C8", "#FFD200"]) # Ghent University colors
        
        plt.title("Vergelijking van gemiddeldes per criterium", fontsize=14, pad=15)
        plt.ylabel("Gemiddelde score")
        plt.xlabel("Criterium")
        plt.ylim(1, 5)  # Scale of criteria
        plt.xticks(rotation=45, ha='right')

        # Display values on bars
        for p in ax.patches:
            height = p.get_height()
            if height > 0:
                ax.annotate(f'{height:.2f}', 
                            (p.get_x() + p.get_width() / 2., height), 
                            ha='center', va='bottom', 
                            fontsize=8, fontweight='bold',
                            xytext=(0, 3), textcoords='offset points')

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, output_score_matrices))
        plt.close()
        
        print(f"Success. Metrics calculated based on {len(data)} rows.")

def main():
    """Iterates over all readability analysis outputs in given directory"""
    for rdbl_file in os.listdir(RDBL_DIR):
        if "Readability analysis" in rdbl_file:
            suffix = rdbl_file.replace("Readability analysis", "").replace(".csv", "")
            if OUTPUT_METRICS + suffix + ".csv" not in os.listdir(OUTPUT_DIR):
                df_check(rdbl_file)
                print(f"Measuring performance for {suffix.replace("_", " ")} Model")
                measure_performance(rdbl_file, OUTPUT_METRICS + suffix + ".csv", OUTPUT_SCORE_MATRICES + suffix + ".svg")
            else:
                print(f"Performance for {suffix.replace("_", " ")} Model has already been measured")
    print("Finished.")

if __name__ == "__main__":
    main()

**Model comparison script:**

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pandas as pd
import os

configuration:

In [4]:
INPUT_DIR = "./Performance measuring"
OUTPUT_DIR = "."
FILE_PATTERN = r"Rdbl performance.*\.csv"
OUTPUT_FILE = "Rdbl model performance comparison"

main algorithm:

In [ ]:
def extract_global_metrics(filepath):
    """
    Extracts global metrics from the performance measure .csv.
    """
    try:
        df = pd.read_csv(filepath)
        filename = os.path.basename(filepath)
        
        summary = df[df['Criteria'] == 'SUBCRITERIA_GLOBAL_METRICS']
        if summary.empty:
            print(f"Warning: No 'SUBCRITERIA_GLOBAL_METRICS' row found in {filename}. Skipping.")
            return None
        
        # Extract MAEm, Krippendorff's Alpha & Average Recall from the 'Note' column using regex
        note = str(summary['Note'].values[0])

        maem_match = re.search(r"MAEm: ([\d\.]+)", note)
        maem = float(maem_match.group(1)) if maem_match else None

        recall_match = re.search(r"Global Avg Recall N/A: ([\d\.]+)", note)
        avg_recall = float(recall_match.group(1)) if recall_match else None

        k_alpha_match = re.search(r"Global Alpha: ([\d\.]+)", note)
        k_alpha = float(k_alpha_match.group(1)) if k_alpha_match else None
        
        # Clean up filename for the legend
        display_name = filename.replace(".csv", "").replace("Rdbl performance", "").lstrip("_").replace("_", " ")
        # Alternative naming to be used in thesis scripture:
        display_name = display_name.replace("role", "rol ").replace("examples", "few-shot").replace("reasoning", "CoT").replace(" rol 0", "")
        display_name = display_name.replace("selfc", "zelfreflectie").replace("GPT5n", "GPT-5-nano").replace("GPT5.1", "GPT-5.1")
        if display_name in ("GPT-5.1", "GPT-5-nano"): display_name += " baseline"
        
        return {
            "Model": display_name,
            "MAEm": maem,
            "Average Recall N/A": avg_recall,
            "Krippendorff's Alpha": k_alpha
        }
    
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None

def main():
   
    # 1. Find all relevant files in the subfolder
    all_files = [f for f in os.listdir(INPUT_DIR) if re.match(FILE_PATTERN, f)]

    if not all_files:
        print("No files found matching the pattern and filters.")
        return

    print(f"Found {len(all_files)} potential files. Extracting subcriterion metrics...")

    # 3. Extract metrics
    comparison_data = []
    for file in all_files:
        filepath = os.path.join(INPUT_DIR, file)
        metrics = extract_global_metrics(filepath)
        if metrics:
            comparison_data.append(metrics)

    if not comparison_data:
        print("No compatible data found.")
        return

    df_comp = pd.DataFrame(comparison_data)

    # 4. Determine the best model
    best_model_idx = df_comp['MAEm'].idxmin()   # General best model is the one with lowest MAEm
    best_model = df_comp.loc[best_model_idx]
    best_k_a_idx = df_comp["Krippendorff's Alpha"].idxmax()    # Model with best Krippendorff's Alpha
    best_k_a = df_comp.loc[best_k_a_idx]
    best_recall_idx = df_comp['Average Recall N/A'].idxmax()    # Model with best average recall
    best_recall = df_comp.loc[best_recall_idx]

    print("\n--- PERFORMANCE RANKING ---")
    print(df_comp.sort_values(by="MAEm", ascending=True).to_string(index=False))
    
    print(f"\nBEST MODEL CONFIGURATION: {best_model['Model']}")
    print(f"Best MAEm : {best_model['MAEm']:.4f}")
    print(f"Best Krippendorff's Alpha : {best_k_a["Krippendorff's Alpha"]:.4f}")
    print(f"Best Average Recall (N/A) : {best_recall['Average Recall N/A']:.4f}")

    df_comp.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_FILE + ".csv"), index=False, encoding='utf-8-sig')
    print(f"\nComparison sheet saved as: {OUTPUT_FILE}.csv")

    # 5. Visualization
    df_melted = df_comp.melt(id_vars="Model", var_name="Metric", value_name="Score")

    plt.figure(figsize=(14, 8))
    sns.set_style("whitegrid")
    
    # Create the plot
    color_palette = ["#1E64C8", "#FFD200", "#8BBEE8"]  # University of Ghent color palette
    ax = sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", palette=color_palette)
    
    # Axis labels are Dutch to use in thesis scripture
    plt.title("Classificatie: modelvergelijking", fontsize=16, pad=20)
    plt.ylabel("Score", fontsize=12)
    plt.xlabel("Modelconfiguratie", fontsize=12)
    plt.xticks(rotation=30, ha='right')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    
    # Add values on top of bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{height:.3f}', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='center', 
                        xytext=(0, 10), 
                        textcoords='offset points',
                        fontsize=9,
                        fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, OUTPUT_FILE + ".svg"))
    print(f"\nComparison graph saved as: {OUTPUT_FILE}.svg")

if __name__ == "__main__":
    main()